# Distil a spam filter

**Large Language Models, Lecture 3.** In class we took a spam filter with 125 million knobs and copied what it knows into a tiny network. Now you try.

You do not need to write any code. Press **Runtime > Run all** first and read the results. Then change the numbers in the next cell, run again, and answer the questions at the end.

The first run takes a few minutes, because it downloads the teacher. We load the teacher's scores on all the messages, which we computed in advance, so you do not need a GPU. Training has some randomness, so your numbers may differ a little from the slides.

**The setup**
- **The teacher:** `mshenoda/roberta-spam`, with about 125 million knobs.
- **The student:** a tiny network that only counts words, with about 16,000 knobs.
- **The data:** 5,574 real text messages, about 13% of them spam.

In [ ]:
# The numbers to play with
N_HUMAN = 200        # how many messages humans label for the student
N_TEACHER = 4074     # how many messages the teacher labels for the student (at most 4074)
TEMPERATURE = 5      # the temperature for soft labels
HIDDEN = 16          # the size of the student's hidden layer
SEEDS = 3            # how many times to repeat each training run

In [ ]:
# Get the messages. You do not need to read this cell.
import io, random, re, urllib.request, zipfile
from collections import Counter

import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
raw = urllib.request.urlopen(url).read()
lines = zipfile.ZipFile(io.BytesIO(raw)).read('SMSSpamCollection').decode('utf-8').splitlines()
rows = [line.split('\t', 1) for line in lines]
texts = [t for _, t in rows]
y = torch.tensor([1 if label == 'spam' else 0 for label, _ in rows])

random.seed(0)
order = list(range(len(texts)))
random.shuffle(order)
test, pool = order[:1500], order[1500:]
print(f'{len(texts)} messages, {y.float().mean():.1%} spam. {len(test)} held out for testing.')
print('Example:', texts[pool[0]])

In [ ]:
# The teacher's scores. Running the teacher on 5,574 messages takes a long time without a GPU,
# so we load scores that we computed in advance. Set this to True to run it yourself.
RUN_TEACHER_YOURSELF = False

import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
name = 'mshenoda/roberta-spam'


def load_teacher():
    tok = AutoTokenizer.from_pretrained(name)
    teacher = AutoModelForSequenceClassification.from_pretrained(name).eval().to(device)
    return tok, teacher


if RUN_TEACHER_YOURSELF:
    tok, teacher = load_teacher()
    logits = []
    with torch.no_grad():
        for i in range(0, len(texts), 32):
            batch = tok(texts[i:i + 32], return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
            logits.append(teacher(**batch).logits.cpu())
    logits = torch.cat(logits)
else:
    scores = 'https://raw.githubusercontent.com/divinites/llm-course-labs/main/teacher_scores.json'
    logits = torch.tensor(json.loads(urllib.request.urlopen(scores).read()))


def score(pred, truth):
    caught = ((pred == 1) & (truth == 1)).sum().item() / max((truth == 1).sum().item(), 1)
    false_alarm = ((pred == 1) & (truth == 0)).sum().item() / max((truth == 0).sum().item(), 1)
    return {'accuracy': (pred == truth).float().mean().item(), 'spam caught': caught, 'false alarms': false_alarm}


results = {'The teacher': score(logits[test].argmax(-1), y[test])}
print(results['The teacher'])

In [ ]:
# The student: count the words, then one small hidden layer.
word = re.compile(r"[a-z0-9'£$]+")
counts = Counter(w for i in pool for w in set(word.findall(texts[i].lower())))
vocab = {w: k for k, (w, _) in enumerate(counts.most_common(1000))}


def features(idx):
    X = torch.zeros(len(idx), len(vocab))
    for row, i in enumerate(idx):
        for w in word.findall(texts[i].lower()):
            if w in vocab:
                X[row, vocab[w]] += 1
    return torch.log1p(X)


def make_student():
    return torch.nn.Sequential(torch.nn.Linear(len(vocab), HIDDEN), torch.nn.ReLU(), torch.nn.Linear(HIDDEN, 2))


Xtest = features(test)


def train(idx, target, T=1.0, seed=0):
    torch.manual_seed(seed)
    student = make_student()
    opt = torch.optim.Adam(student.parameters(), lr=.01, weight_decay=1e-4)
    X = features(idx)
    for _ in range(300):
        opt.zero_grad()
        out = student(X)
        if target.dim() == 1:          # hard labels: spam or not
            loss = F.cross_entropy(out, target)
        else:                          # soft labels: the teacher's scores, softened by T
            loss = F.kl_div(F.log_softmax(out / T, -1), F.softmax(target / T, -1), reduction='batchmean') * T * T
        loss.backward()
        opt.step()
    return student


def average(idx, target, T=1.0):
    runs = []
    for seed in range(SEEDS):
        student = train(idx, target, T, seed)
        with torch.no_grad():
            runs.append(score(student(Xtest).argmax(-1), y[test]))
    return {k: sum(r[k] for r in runs) / len(runs) for k in runs[0]}


print('Student knobs:', f'{sum(p.numel() for p in make_student().parameters()):,}')

In [ ]:
# Train the student in four ways, and compare.
human, more = pool[:N_HUMAN], pool[:N_TEACHER]
results[f'Student: {N_HUMAN} human labels'] = average(human, y[human])
results[f"Student: teacher's answers on {N_TEACHER} (black box)"] = average(more, logits[more].argmax(-1))
results[f"Student: teacher's soft labels on {N_TEACHER}, T={TEMPERATURE} (white box)"] = average(more, logits[more], T=TEMPERATURE)
results[f'Student: {N_TEACHER} human labels'] = average(more, y[more])

always_not_spam = torch.zeros(len(test), dtype=torch.long)
results['The laziest model: always say not spam'] = score(always_not_spam, y[test])

print(f"{'':62}{'accuracy':>10}{'caught':>10}{'false alarms':>14}")
for k, v in results.items():
    print(f"{k:62}{v['accuracy']:>10.1%}{v['spam caught']:>10.1%}{v['false alarms']:>14.1%}")

In [ ]:
# Try your own emails. Change these and run the cell again.
my_emails = [
    'Claim your free prize right now',
    'Feel free to call me anytime',
    'Send me your password now',
    'Your lunch is at noon',
]

tok, teacher = load_teacher()
student = train(more, logits[more].argmax(-1), seed=0)
batch = tok(my_emails, return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    teacher_p = teacher(**batch).logits.softmax(-1)[:, 1].cpu()
X = torch.zeros(len(my_emails), len(vocab))
for row, text in enumerate(my_emails):
    for w in word.findall(text.lower()):
        if w in vocab:
            X[row, vocab[w]] += 1
with torch.no_grad():
    student_p = student(torch.log1p(X)).softmax(-1)[:, 1]
for text, t, s in zip(my_emails, teacher_p, student_p):
    print(f'teacher {t:.2f}   student {s:.2f}   {text}')

## Your turn

Change the numbers near the top, run everything again, and try to answer these. Some of them have a trap.

1. **Labels.** Our student caught 44% of the spam with 200 human labels. How many human labels does it take to catch 80%? Now look at the cost: how much would it cost you to pay someone to label them, and what did the teacher's labels cost?
2. **Temperature.** Try `TEMPERATURE` = 1, 2 and 20. Can you make the soft labels beat the teacher's plain answers? Why might soft labels help less here, with two classes, than with thousands?
3. **The laziest model.** What accuracy does it score? So which column should your manager be looking at?
4. **Size.** Try `HIDDEN` = 2 and 256. How much does a bigger student help? What does that tell you about where its limits come from?
5. **Fool it.** In the last cell, write an email that the student calls safe but the teacher calls spam. Then write one that fools both. What did you learn about what each of them is really looking at?
6. **Beat the teacher?** The student never beats the teacher here. In class we saw cases where a student does. What is different about those cases?

**Careful about one thing.** The teacher's model card says that its training data includes this very collection of messages. So it has seen most of them before, and its score here is too kind. Its answers may also be close to the human labels simply because it remembers them. That makes the student's result look better than a truly independent teacher would. Everything you measure about the student on the held-out messages is still fair, because the student has never seen them.

**Data:** UCI SMS Spam Collection (Almeida, Gómez Hidalgo and Yamakami, 2011), CC BY 4.0. It is downloaded when you run the notebook.